In [26]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
from scipy.signal import butter, filtfilt, iirnotch

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [18]:
df = pd.read_csv("../data/Cosmic_Horror/baseline_20260226_183434.csv", skiprows=8)

In [20]:
def bandpass_filter(signal, fs, low=20, high=450, order=4):
    nyq = 0.5 * fs
    if high >= nyq:
        high = nyq * 0.95  # safe margin
    b, a = butter(order, [low/nyq, high/nyq], btype='band')
    return filtfilt(b, a, signal)

def notch_filter(signal, fs, freq=60, Q=30):
    nyq = 0.5 * fs
    b, a = iirnotch(freq/nyq, Q)
    return filtfilt(b, a, signal)

# --- Preprocessing for DataFrame ---
def preprocess_emg_df(df, fs, apply_notch=False):
    """
    df: pandas DataFrame with columns: timestamp, Mock_Ch1, Mock_Ch2, ...
    fs: sampling rate (Hz)
    apply_notch: whether to apply 60 Hz notch
    Returns: DataFrame with filtered EMG channels (timestamp preserved)
    """
    df_filtered = df.copy()

    # Assume first column is timestamp
    channels = df.columns[1:]

    for ch in channels:
        signal = df[ch].values
        signal = bandpass_filter(signal, fs)
        if apply_notch:
            signal = notch_filter(signal, fs)
        df_filtered[ch] = signal

    return df_filtered

preprocessed = preprocess_emg_df(df, 250, True)
preprocessed

,timestamp,Mock_Ch1,Mock_Ch2,Mock_Ch3,Mock_Ch4
0,1.772149e+09,250.125884,-19.238535,-70.346016,1.861917
1,1.772149e+09,1458.544072,-111.318356,-1700.759626,96.440490
2,1.772149e+09,1130.254785,42.337308,-452.380649,26.758486
3,1.772149e+09,885.934292,-1177.307018,-2340.502053,-68.990384
4,1.772149e+09,-857.379216,462.149240,480.333545,31.548123
...,...,...,...,...,...
2019,1.772149e+09,-29.036265,-86.461931,155.117383,-22.759272
2020,1.772149e+09,25.435759,-75.756177,-279.065415,-218.917406
2021,1.772149e+09,28.762779,96.622397,547.856356,2177.342077
2022,1.772149e+09,-28.198987,37.357158,209.798825,1510.771058


In [23]:
def extract_features_window(window, channel_names):
    features = {}
    for i, ch in enumerate(channel_names):
        sig = window[:, i]
        features[f"RMS_{ch}"] = np.sqrt(np.mean(sig**2))
        features[f"MAV_{ch}"] = np.mean(np.abs(sig))
        features[f"Var_{ch}"] = np.var(sig)
        features[f"WL_{ch}"] = np.sum(np.abs(np.diff(sig)))
        features[f"ZC_{ch}"] = np.sum(np.diff(np.sign(sig)) != 0)
    return features

# --- Windowing + feature extraction ---
def create_feature_df(df, fs, window_ms=250, overlap=0.5, labels=None):
    """
    df: filtered EMG DataFrame (timestamp + channels)
    fs: sampling rate
    labels: optional array of labels per window
    """
    channels = df.columns[1:]
    data = df[channels].values
    window_size = int(fs * window_ms / 1000)
    step = int(window_size * (1-overlap))

    feature_rows = []
    for start in range(0, len(data) - window_size, step):
        window = data[start:start+window_size, :]
        features = extract_features_window(window, channels)
        feature_rows.append(features)

    feature_df = pd.DataFrame(feature_rows)

    if labels is not None:
        feature_df["Label"] = labels[:len(feature_df)]

    return feature_df

feature_df = create_feature_df(preprocessed, 250)
feature_df

,RMS_Mock_Ch1,MAV_Mock_Ch1,Var_Mock_Ch1,WL_Mock_Ch1,ZC_Mock_Ch1,RMS_Mock_Ch2,MAV_Mock_Ch2,Var_Mock_Ch2,WL_Mock_Ch2,ZC_Mock_Ch2,RMS_Mock_Ch3,MAV_Mock_Ch3,Var_Mock_Ch3,WL_Mock_Ch3,ZC_Mock_Ch3,RMS_Mock_Ch4,MAV_Mock_Ch4,Var_Mock_Ch4,WL_Mock_Ch4,ZC_Mock_Ch4
0,1406.935052,1143.330073,1.976531e+06,105804.516878,34,1179.463485,919.665732,1.390441e+06,82809.471880,33,908.929812,446.873333,8.246213e+05,41893.929375,29,39.301525,30.537811,1.543201e+03,2894.697940,36
1,1526.174330,1210.995216,2.326424e+06,116535.051696,36,1128.575646,807.610531,1.273177e+06,68534.894025,27,61.505930,50.012335,3.782963e+03,4929.275034,36,43.506750,34.691937,1.890490e+03,3125.182612,34
2,1394.474611,1122.960252,1.944476e+06,94625.683143,30,604.673013,257.839523,3.654192e+05,25555.703932,31,52.257456,41.875411,2.730760e+03,4211.480327,35,39.741659,32.639376,1.578287e+03,2830.385173,32
3,1388.940057,1100.591593,1.929140e+06,96336.949129,31,51.443469,40.032860,2.646141e+03,3845.029827,34,46.981781,35.218140,2.206780e+03,3545.611338,36,37.785415,30.123802,1.427142e+03,2852.032391,33
4,1196.602951,848.101473,1.431651e+06,76857.305780,30,50.261190,41.503772,2.522957e+03,3711.867829,31,42.675565,32.375770,1.819812e+03,3067.131105,37,42.077410,33.198359,1.768985e+03,3154.487179,34
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,360.710879,152.458311,1.301093e+05,15427.111460,41,1294.460242,1038.301087,1.675098e+06,107796.107409,36,1385.499961,1093.336918,1.916702e+06,104229.007038,33,96.636506,72.586032,9.337836e+03,7289.373022,36
60,40.637962,32.766699,1.648203e+03,3304.737534,42,1226.309437,978.250702,1.503834e+06,97761.285359,37,1434.100401,1134.586307,2.055457e+06,106272.019731,36,776.328815,422.690073,6.026829e+05,41395.580904,37
61,42.122048,34.207317,1.774257e+03,3541.283929,42,1121.487813,887.894088,1.257690e+06,83779.281027,33,1164.368039,923.693593,1.354582e+06,79110.695460,32,1208.042252,879.427839,1.459352e+06,94372.706068,37
62,46.223612,38.364812,2.136146e+03,3660.623086,39,736.648677,453.736137,5.426506e+05,41464.693667,28,1090.564756,883.148573,1.186358e+06,72371.521171,30,1241.827800,967.147474,1.542024e+06,99141.406996,38


In [25]:
def pipeline(csv, fs, window_size, overlap, label):
    pipeline_df = pd.read_csv(csv, skiprows=8)
    preprocessed_df = preprocess_emg_df(pipeline_df, 250, True)
    pipeline_feature_df = create_feature_df(preprocessed_df, fs, window_size, overlap)
    pipeline_feature_df['label'] = label
    return pipeline_feature_df

result = pipeline("../data/Cosmic_Horror/baseline_20260226_183434.csv", 250, 250, 0.5, "Mock Baseline")
result

,RMS_Mock_Ch1,MAV_Mock_Ch1,Var_Mock_Ch1,WL_Mock_Ch1,ZC_Mock_Ch1,RMS_Mock_Ch2,MAV_Mock_Ch2,Var_Mock_Ch2,WL_Mock_Ch2,ZC_Mock_Ch2,...,MAV_Mock_Ch3,Var_Mock_Ch3,WL_Mock_Ch3,ZC_Mock_Ch3,RMS_Mock_Ch4,MAV_Mock_Ch4,Var_Mock_Ch4,WL_Mock_Ch4,ZC_Mock_Ch4,label
0,1406.935052,1143.330073,1.976531e+06,105804.516878,34,1179.463485,919.665732,1.390441e+06,82809.471880,33,...,446.873333,8.246213e+05,41893.929375,29,39.301525,30.537811,1.543201e+03,2894.697940,36,Mock Baseline
1,1526.174330,1210.995216,2.326424e+06,116535.051696,36,1128.575646,807.610531,1.273177e+06,68534.894025,27,...,50.012335,3.782963e+03,4929.275034,36,43.506750,34.691937,1.890490e+03,3125.182612,34,Mock Baseline
2,1394.474611,1122.960252,1.944476e+06,94625.683143,30,604.673013,257.839523,3.654192e+05,25555.703932,31,...,41.875411,2.730760e+03,4211.480327,35,39.741659,32.639376,1.578287e+03,2830.385173,32,Mock Baseline
3,1388.940057,1100.591593,1.929140e+06,96336.949129,31,51.443469,40.032860,2.646141e+03,3845.029827,34,...,35.218140,2.206780e+03,3545.611338,36,37.785415,30.123802,1.427142e+03,2852.032391,33,Mock Baseline
4,1196.602951,848.101473,1.431651e+06,76857.305780,30,50.261190,41.503772,2.522957e+03,3711.867829,31,...,32.375770,1.819812e+03,3067.131105,37,42.077410,33.198359,1.768985e+03,3154.487179,34,Mock Baseline
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,360.710879,152.458311,1.301093e+05,15427.111460,41,1294.460242,1038.301087,1.675098e+06,107796.107409,36,...,1093.336918,1.916702e+06,104229.007038,33,96.636506,72.586032,9.337836e+03,7289.373022,36,Mock Baseline
60,40.637962,32.766699,1.648203e+03,3304.737534,42,1226.309437,978.250702,1.503834e+06,97761.285359,37,...,1134.586307,2.055457e+06,106272.019731,36,776.328815,422.690073,6.026829e+05,41395.580904,37,Mock Baseline
61,42.122048,34.207317,1.774257e+03,3541.283929,42,1121.487813,887.894088,1.257690e+06,83779.281027,33,...,923.693593,1.354582e+06,79110.695460,32,1208.042252,879.427839,1.459352e+06,94372.706068,37,Mock Baseline
62,46.223612,38.364812,2.136146e+03,3660.623086,39,736.648677,453.736137,5.426506e+05,41464.693667,28,...,883.148573,1.186358e+06,72371.521171,30,1241.827800,967.147474,1.542024e+06,99141.406996,38,Mock Baseline


In [28]:
csv_dict = {
    "arm_down": "../data/Cosmic_Horror/arm down_20260226_201414.csv",
    "arm_up": "../data/Cosmic_Horror/arm up side palm down_20260226_201448.csv",
    "fist_down_out": "../data/Cosmic_Horror/fist-down-out_20260226_201320.csv",
    "fist_down_up": "../data/Cosmic_Horror/fist-down-up_20260226_201341.csv",
    "palm_down_out": "../data/Cosmic_Horror/palm-down-out_20260226_201210.csv",
    "palm_down_up": "../data/Cosmic_Horror/palm-down-up_20260226_201238.csv",
    "palm_up_out": "../data/Cosmic_Horror/palm-up-out_20260226_200912.csv",
    "peace_out": "../data/Cosmic_Horror/peace out_20260226_201518.csv"
}

def batch_pipeline(csv_dictionary: dict, fs, window_size, overlap):
    df_batch = pd.DataFrame()

    for key, val in csv_dictionary.items():
        csv_df = pipeline(val, fs, window_size, overlap, key)
        df_batch = pd.concat([df_batch, csv_df])

    return df_batch

batch_df = batch_pipeline(csv_dict, 250, 250, 0.5)
batch_df


,RMS_GSR,MAV_GSR,Var_GSR,WL_GSR,ZC_GSR,RMS_Ch2,MAV_Ch2,Var_Ch2,WL_Ch2,ZC_Ch2,label
0,0.262145,0.190444,0.068697,12.198558,16,0.262223,0.210532,0.068749,18.754300,33,arm_down
1,0.179680,0.123718,0.032236,8.322679,18,0.274912,0.238019,0.075530,20.907075,30,arm_down
2,0.130360,0.084928,0.016991,4.904283,17,0.252384,0.210607,0.063696,19.834861,34,arm_down
3,0.187544,0.133666,0.035166,7.414677,16,0.259309,0.207926,0.067234,20.270626,36,arm_down
4,0.207863,0.156828,0.043203,8.827907,14,0.265048,0.225898,0.070249,21.242172,33,arm_down
...,...,...,...,...,...,...,...,...,...,...,...
78,0.377926,0.292846,0.142762,16.824215,17,0.234936,0.171928,0.055195,11.874169,26,peace_out
79,0.751307,0.556176,0.564283,34.086259,17,0.207154,0.136809,0.042911,10.140053,26,peace_out
80,0.757249,0.569749,0.573424,34.200088,15,0.188174,0.113943,0.035402,9.047159,25,peace_out
81,0.737224,0.532621,0.542092,34.661331,20,0.274386,0.189608,0.075288,13.774962,26,peace_out
